**Análisis: KPI´S**

In [1]:
import pandas as pd
import os

# Construir ruta relativa
ruta = os.path.join('..', 'Data', 'TouristAccommodationClean26012026.csv')

# Leer CSV con encoding correcto
df = pd.read_csv(ruta, encoding='utf-8-sig')

# Ver primeras filas
df.head()

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,amenities_count,month,year,month_new
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,NaN,Private room,2,2.0,1.0,...,100.0,False,75.0,spain,Malaga,2018-07-31,19.0,2018-07,2018,JUL
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1.0,1.0,...,90.0,False,52.0,spain,Madrid,2020-01-10,31.0,2020-01,2020,JAN
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1.0,2.0,...,100.0,True,142.0,spain,Sevilla,2019-07-29,34.0,2019-07,2019,JUL
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2.0,1.0,...,90.0,True,306.0,spain,Barcelona,2020-01-10,27.0,2020-01,2020,JAN
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,NaN,Private room,5,1.0,2.0,...,100.0,False,39.0,spain,Girona,2019-02-19,43.0,2019-02,2019,FEB


In [2]:
df.columns

Index(['apartment_id', 'name', 'description', 'host_id', 'neighbourhood_name',
       'neighbourhood_district', 'room_type', 'accommodates', 'bathrooms',
       'bedrooms', 'beds', 'amenities_list', 'price', 'minimum_nights',
       'maximum_nights', 'has_availability', 'availability_30',
       'availability_60', 'availability_90', 'availability_365',
       'number_of_reviews', 'first_review_date', 'last_review_date',
       'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'is_instant_bookable', 'reviews_per_month',
       'country', 'city', 'insert_date', 'amenities_count', 'month', 'year',
       'month_new'],
      dtype='object')

**Occupancy Rate = (Number of Occupied Days / Number of Days) × 100 /tasa de ocupación**

In [3]:
# Tasa de ocupacion 
for period in ['30', '60', '90', '365']:
    print(f'\nOccupancy Rate {period} days:')
    print(
        ((int(period) - df[f"availability_{period}"]) / int(period)).mean()*100) #calcula dias ocupados para cada alojamiento/divide por el 
                                                                                 #total de dias
                                                                                 #mean calcula la media de los alojamientos


Occupancy Rate 30 days:
58.695833333333326

Occupancy Rate 60 days:
53.98875000000001

Occupancy Rate 90 days:
50.44499999999999

Occupancy Rate 365 days:
48.4011301369863


**Promedio de ocupación por 30 dias de todos los alojamientos**

In [4]:
df["occupancy_rate_30"] = ((30 - df["availability_30"]) / 30) * 100

In [5]:
promedio_30 = df["occupancy_rate_30"].mean()
print(f"Promedio de ocupación 30 días: {promedio_30:.2f}%")

Promedio de ocupación 30 días: 58.70%


In [6]:
# Promedio de ocupación por ciudad para 30 días
promedio_ciudad_30 = df.groupby("city")["occupancy_rate_30"].mean()
promedio_ciudad_30.round(2)

city
Barcelona    62.66
Girona       51.27
Madrid       64.19
Malaga       58.21
Mallorca     54.92
Menorca      51.89
Sevilla      52.83
Valencia     56.95
Name: occupancy_rate_30, dtype: float64

**Ciudad con mayor ocupacion**

In [7]:
# Ciudad con mayor ocupación
ciudad_mayor_ocupacion = promedio_ciudad_30.idxmax()
valor_mayor_ocupacion = promedio_ciudad_30.max()

print(f"Ciudad con mayor ocupación promedio (30 días): {ciudad_mayor_ocupacion}")
print(f"Ocupación promedio: {valor_mayor_ocupacion:.2f}%")

Ciudad con mayor ocupación promedio (30 días): Madrid
Ocupación promedio: 64.19%


**Indice de satisfacción general**

In [8]:
indice_satisfaccion = (df["review_scores_rating"] / 10).mean()
print(round(indice_satisfaccion, 2))

91.97


**Item con mayor satisfacción promedio**

In [9]:
promedios = df[[
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location"
]].mean().round(2)

print(promedios)

review_scores_accuracy         94.56
review_scores_cleanliness      93.16
review_scores_checkin          96.28
review_scores_communication    96.35
review_scores_location         95.32
dtype: float64


In [10]:
item_top = promedios.idxmax()
valor_top = promedios.max()

print(f"Ítem con mayor satisfacción promedio: {item_top} ({valor_top})")

Ítem con mayor satisfacción promedio: review_scores_communication (96.35)


In [11]:
columns = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location"
]

# Agrupar por ciudad y calcular promedio de cada métrica
avg_scores = df.groupby("city")[columns].mean()

# Crear columna de satisfacción compuesta
avg_scores["satisfaccion_compuesta"] = avg_scores.mean(axis=1)

# Ciudad con mayor satisfacción
top_city = avg_scores["satisfaccion_compuesta"].idxmax()
top_score = avg_scores["satisfaccion_compuesta"].max()

print(f"Ciudad con mayor satisfacción promedio: {top_city} ({top_score:.2f})")

avg_scores.sort_values("satisfaccion_compuesta", ascending=False)

Ciudad con mayor satisfacción promedio: Sevilla (96.91)


,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,satisfaccion_compuesta
city,,,,,,
Sevilla,96.820513,96.000000,97.641026,97.435897,96.666667,96.912821
Madrid,95.231092,93.781513,96.643308,96.862745,96.496146,95.802961
Menorca,96.335878,92.213740,96.821705,97.786260,95.116279,95.654772
Valencia,95.710145,93.275362,96.840580,96.927536,95.304348,95.611594
Malaga,95.327869,94.344262,96.612022,96.775956,94.590164,95.530055
Mallorca,94.804469,94.491620,96.759777,96.729911,94.352679,95.427691
Barcelona,93.363017,92.014652,95.655136,95.515437,95.165181,94.342685
Girona,93.718412,91.568862,95.457831,95.841346,94.427021,94.202694
